# 02 原始财务数据清洗

## 目标

在保留 `data/raw` 原始数据不变的前提下，根据上一阶段数据审计结果，
逐步建立明确、可复现的数据清洗规则，并最终输出标准化的 processed 数据。

本 Notebook 坚持：

1. 不直接修改 raw 文件；
2. 每项清洗先说明规则，再执行转换；
3. 清洗前后进行核对；
4. 对无法机械判断的问题暂不擅自删除。

## 第一阶段：股票代码标准化

`stock_code` 是公司标识符，而不是用于数学计算的数值变量。

目标格式：

- 字符串类型；
- 去除首尾空格；
- 非缺失代码统一为 6 位；
- 缺失值继续保留为缺失值；
- 不通过数值运算处理股票代码。

In [43]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

assert (PROJECT_ROOT / "pyproject.toml").exists()
assert DATA_RAW.exists()

print("Project paths initialized successfully.")

assert (PROJECT_ROOT / "pyproject.toml").exists()
assert DATA_RAW.exists()

Project paths initialized successfully.


In [2]:
raw_financials = pd.read_excel(
    DATA_RAW / "firm_financials.xlsx",
    dtype={"stock_code": "string"},
)

raw_financials.shape

(260, 11)

In [3]:
print(raw_financials["stock_code"].dtype)

raw_financials[["stock_code"]].head(30)

string


,stock_code
0,000001
1,000001
2,000001
3,000001
4,000001
5,000001
6,000002
7,2
8,000002
9,000002


In [4]:
raw_financials["stock_code"].map(repr).head(30)

0     ' 000001 '
1       '000001'
2       '000001'
3       '000001'
4       '000001'
5       '000001'
6       '000002'
7            '2'
8       '000002'
9       '000002'
10      '000002'
11      '000002'
12      '000003'
13      '000003'
14           '3'
15      '000003'
16      '000003'
17      '000003'
18      '000004'
19      '000004'
20      '000004'
21          <NA>
22      '000004'
23      '000004'
24      '000005'
25      '000005'
26      '000005'
27      '000005'
28     '000005 '
29      '000005'
Name: stock_code, dtype: str

In [5]:
financials = raw_financials.copy()

print("raw_financials shape:", raw_financials.shape)
print("financials shape:", financials.shape)

raw_financials shape: (260, 11)
financials shape: (260, 11)


In [6]:
stock_code_stripped = financials["stock_code"].str.strip()

stock_code_stripped.map(repr).head(30)

0     '000001'
1     '000001'
2     '000001'
3     '000001'
4     '000001'
5     '000001'
6     '000002'
7          '2'
8     '000002'
9     '000002'
10    '000002'
11    '000002'
12    '000003'
13    '000003'
14         '3'
15    '000003'
16    '000003'
17    '000003'
18    '000004'
19    '000004'
20    '000004'
21        <NA>
22    '000004'
23    '000004'
24    '000005'
25    '000005'
26    '000005'
27    '000005'
28    '000005'
29    '000005'
Name: stock_code, dtype: str

In [7]:
invalid_stock_code_mask = (
    stock_code_stripped.notna()
    & ~stock_code_stripped.str.fullmatch(r"\d+")
)

invalid_stock_codes = stock_code_stripped[invalid_stock_code_mask]

print("非数字股票代码数量:", len(invalid_stock_codes))
invalid_stock_codes

非数字股票代码数量: 0


Series([], Name: stock_code, dtype: string)

In [8]:
stock_code_length_counts = (
    stock_code_stripped
    .dropna()
    .str.len()
    .value_counts()
    .sort_index()
)

stock_code_length_counts

stock_code
1      2
6    257
Name: count, dtype: Int64

In [9]:
stock_code_clean = stock_code_stripped.str.zfill(6)

stock_code_clean.map(repr).head(30)

0     '000001'
1     '000001'
2     '000001'
3     '000001'
4     '000001'
5     '000001'
6     '000002'
7     '000002'
8     '000002'
9     '000002'
10    '000002'
11    '000002'
12    '000003'
13    '000003'
14    '000003'
15    '000003'
16    '000003'
17    '000003'
18    '000004'
19    '000004'
20    '000004'
21        <NA>
22    '000004'
23    '000004'
24    '000005'
25    '000005'
26    '000005'
27    '000005'
28    '000005'
29    '000005'
Name: stock_code, dtype: str

In [10]:
before_missing = raw_financials["stock_code"].isna().sum()
after_missing = stock_code_clean.isna().sum()

invalid_clean_code_mask = (
    stock_code_clean.notna()
    & ~stock_code_clean.str.fullmatch(r"\d{6}")
)

print("清洗前缺失数量:", before_missing)
print("清洗后缺失数量:", after_missing)
print("清洗后不符合6位纯数字规则的数量:", invalid_clean_code_mask.sum())

stock_code_clean.dropna().str.len().value_counts().sort_index()

清洗前缺失数量: 1
清洗后缺失数量: 1
清洗后不符合6位纯数字规则的数量: 0


stock_code
6    259
Name: count, dtype: Int64

In [11]:
same_stock_code_mask = (
    raw_financials["stock_code"].eq(stock_code_clean).fillna(False)
    | (
        raw_financials["stock_code"].isna()
        & stock_code_clean.isna()
    )
)

stock_code_changes = pd.DataFrame(
    {
        "before": raw_financials["stock_code"].map(repr),
        "after": stock_code_clean.map(repr),
    }
)

stock_code_changes = stock_code_changes.loc[~same_stock_code_mask]

print("实际发生变化的记录数:", len(stock_code_changes))
stock_code_changes

实际发生变化的记录数: 5


,before,after
0,' 000001 ','000001'
7,'2','000002'
14,'3','000003'
28,'000005 ','000005'
240,' 000001 ','000001'


In [12]:
assert before_missing == after_missing
assert invalid_clean_code_mask.sum() == 0
assert stock_code_clean.dropna().str.len().eq(6).all()

financials["stock_code"] = stock_code_clean

print("stock_code 标准化完成")
print(financials["stock_code"].dtype)

stock_code 标准化完成
string


In [13]:
print("year dtype:", financials["year"].dtype)

financials["year"].map(type).value_counts()

year dtype: object


year
<class 'int'>    250
<class 'str'>     10
Name: count, dtype: int64

In [14]:
financials["year"].value_counts(dropna=False).sort_index(key=lambda x: x.astype(str))

year
2020     43
2021     43
2022     39
2022      4
2023     38
2023年     6
2024     44
2025     43
Name: count, dtype: int64

In [15]:
year_text = financials["year"].astype("string").str.strip()

year_text.value_counts(dropna=False).sort_index()

year
2020     43
2021     43
2022     43
2023     38
2023年     6
2024     44
2025     43
Name: count, dtype: int64[pyarrow]

In [16]:
invalid_year_mask = (
    year_text.notna()
    & ~year_text.str.fullmatch(r"\d{4}")
)

invalid_years = year_text[invalid_year_mask]

print("非标准年份记录数:", len(invalid_years))
invalid_years.value_counts(dropna=False)

非标准年份记录数: 6


year
2023年    6
Name: count, dtype: int64[pyarrow]

In [17]:
year_clean_text = year_text.str.replace(r"年$", "", regex=True)

year_clean_text.value_counts(dropna=False).sort_index()

year
2020    43
2021    43
2022    43
2023    44
2024    44
2025    43
Name: count, dtype: int64[pyarrow]

In [18]:
invalid_year_after_rule = (
    year_clean_text.notna()
    & ~year_clean_text.str.fullmatch(r"\d{4}")
)

print(
    "应用年份格式规则后，仍非4位纯数字的记录数:",
    invalid_year_after_rule.sum(),
)

year_clean_text[invalid_year_after_rule]

应用年份格式规则后，仍非4位纯数字的记录数: 0


Series([], Name: year, dtype: string)

In [19]:
year_clean = pd.to_numeric(
    year_clean_text,
    errors="raise",
).astype("Int64")

print("清洗后 year dtype:", year_clean.dtype)

year_clean.value_counts(dropna=False).sort_index()

清洗后 year dtype: Int64


year
2020    43
2021    43
2022    43
2023    44
2024    44
2025    43
Name: count, dtype: Int64

In [20]:
year_missing_before = raw_financials["year"].isna().sum()
year_missing_after = year_clean.isna().sum()

year_out_of_range_mask = (
    year_clean.notna()
    & ~year_clean.between(2020, 2025)
)

print("清洗前年份缺失数量:", year_missing_before)
print("清洗后年份缺失数量:", year_missing_after)
print("超出 2020-2025 范围的年份数量:", year_out_of_range_mask.sum())
print("最小年份:", year_clean.min())
print("最大年份:", year_clean.max())

清洗前年份缺失数量: 0
清洗后年份缺失数量: 0
超出 2020-2025 范围的年份数量: 0
最小年份: 2020
最大年份: 2025


In [21]:
year_before_text = raw_financials["year"].astype("string")

year_same_mask = (
    year_before_text.eq(year_clean.astype("string")).fillna(False)
    | (
        raw_financials["year"].isna()
        & year_clean.isna()
    )
)

year_changes = pd.DataFrame(
    {
        "before": raw_financials["year"].map(repr),
        "after": year_clean.map(repr),
    }
).loc[~year_same_mask]

print("年份表示实际发生变化的记录数:", len(year_changes))
year_changes

年份表示实际发生变化的记录数: 6


,before,after
3,'2023年',2023
9,'2023年',2023
15,'2023年',2023
21,'2023年',2023
27,'2023年',2023
251,'2023年',2023


In [22]:
assert invalid_year_after_rule.sum() == 0
assert year_missing_before == year_missing_after
assert year_out_of_range_mask.sum() == 0
assert year_clean.dropna().between(2020, 2025).all()

financials["year"] = year_clean

print("year 标准化完成")
print("year dtype:", financials["year"].dtype)

year 标准化完成
year dtype: Int64


## 第二阶段：firm-year 主键重复检查

在完成 `stock_code` 和 `year` 标准化后，
重新检查 `stock_code + year` 是否能够唯一标识一条公司年度记录。

本阶段只识别和分类重复问题，不直接删除记录。

In [23]:
key_cols = ["stock_code", "year"]

firm_year_duplicate_mask = financials.duplicated(
    subset=key_cols,
    keep=False,
)

duplicate_firm_year_rows = financials.loc[
    firm_year_duplicate_mask
].copy()

print(
    "位于重复 firm-year 组中的记录数:",
    firm_year_duplicate_mask.sum(),
)

duplicate_group_count = (
    duplicate_firm_year_rows
    .groupby(key_cols, dropna=False)
    .ngroups
)

print(
    "重复 firm-year 组数:",
    duplicate_group_count,
)

位于重复 firm-year 组中的记录数: 40
重复 firm-year 组数: 20


In [24]:
duplicate_group_sizes = (
    duplicate_firm_year_rows
    .groupby(key_cols, dropna=False)
    .size()
    .reset_index(name="row_count")
    .sort_values(
        ["row_count", "stock_code", "year"],
        ascending=[False, True, True],
    )
)

duplicate_group_sizes

,stock_code,year,row_count
0,000001,2020,2
1,000002,2024,2
2,000003,2025,2
3,000005,2023,2
4,000006,2024,2
5,000008,2022,2
6,000009,2023,2
7,000011,2021,2
8,000012,2022,2
9,000014,2020,2


In [25]:
duplicate_firm_year_rows = (
    duplicate_firm_year_rows
    .sort_values(key_cols)
)

duplicate_firm_year_rows[
    [
        "stock_code",
        "company_name",
        "year",
        "total_assets",
        "total_liabilities",
        "revenue",
        "net_profit",
        "cash",
        "rd_expense",
        "roe",
        "employees",
    ]
]

,stock_code,company_name,year,total_assets,total_liabilities,revenue,net_profit,cash,rd_expense,roe,employees
0,000001,华辰科技股份有限公司,2020,44813001602,23908847993,8788852319,8.233325e+08,4.322096e+09,655221236,0.0394,14725.0
240,000001,华辰科技股份有限公司,2020,44813001602,23908847993,8788852319,8.233325e+08,4.322096e+09,655221236,0.0394,14725.0
10,000002,新岳科技股份有限公司,2024,9274052273,2248802709,44523089214,1.471324e+10,1.739329e+09,2588058155,2.0943,36975.0
250,000002,新岳科技股份有限公司,2024,9274052273,2248802709,44523089214,1.500893e+09,1.739329e+09,2588058155,2.0943,36975.0
17,000003,海川科技股份有限公司,2025,51903132738,21094446603,6197206185,1.456729e+08,1.501682e+10,367316599,0.0047,17290.0
241,000003,海川科技股份有限公司,2025,51903132738,21094446603,6197206185,1.456729e+08,1.501682e+10,367316599,0.0047,17290.0
27,000005,宏远科技股份有限公司,2023,33191213417,20487864880,25711723804,2.855787e+09,3.907137e+09,1678145406,0.2248,3315.0
251,000005,宏远科技股份有限公司,2023,33191213417,20487864880,25711723804,1.766409e+09,3.907137e+09,1678145406,0.2248,3315.0
34,000006,智恒科技股份有限公司,2024,24192980006,8900245804,24585101604,4.884060e+09,5.243925e+09,390759665,0.3194,44831.0
242,000006,智恒科技股份有限公司,2024,24192980006,8900245804,24585101604,4.884060e+09,5.243925e+09,390759665,0.3194,44831.0


In [26]:
print(
    "完全重复的后续记录数:",
    financials.duplicated(keep="first").sum(),
)

print(
    "位于完全重复组中的全部记录数:",
    financials.duplicated(keep=False).sum(),
)

完全重复的后续记录数: 10
位于完全重复组中的全部记录数: 20


## 第三阶段：重复 firm-year 分类

重复的 `stock_code + year` 组合需要进一步区分：

- **完全重复**：同一 firm-year 的所有字段完全一致；
- **冲突重复**：firm-year 相同，但一个或多个变量取值不同。

完全重复通常可以安全去除多余副本；
冲突重复则不能机械删除，需要先识别冲突字段并决定处理规则。

In [27]:
non_key_cols = [
    col
    for col in financials.columns
    if col not in key_cols
]

group_variation = (
    duplicate_firm_year_rows
    .groupby(key_cols, dropna=False)[non_key_cols]
    .nunique(dropna=False)
)

group_variation

,,company_name,total_assets,total_liabilities,revenue,net_profit,cash,rd_expense,roe,employees
stock_code,year,,,,,,,,,
000001,2020,1,1,1,1,1,1,1,1,1
000002,2024,1,1,1,1,2,1,1,1,1
000003,2025,1,1,1,1,1,1,1,1,1
000005,2023,1,1,1,1,2,1,1,1,1
000006,2024,1,1,1,1,1,1,1,1,1
000008,2022,1,1,1,1,2,1,1,1,1
000009,2023,1,1,1,1,1,1,1,1,1
000011,2021,1,1,1,1,2,1,1,1,1
000012,2022,1,1,1,1,1,1,1,1,1


In [28]:
duplicate_group_classification = pd.DataFrame(
    index=group_variation.index
)

duplicate_group_classification["max_distinct_values"] = (
    group_variation.max(axis=1)
)

duplicate_group_classification["duplicate_type"] = (
    duplicate_group_classification["max_distinct_values"]
    .eq(1)
    .map(
        {
            True: "exact",
            False: "conflict",
        }
    )
)

duplicate_group_classification.reset_index()

,stock_code,year,max_distinct_values,duplicate_type
0,000001,2020,1,exact
1,000002,2024,2,conflict
2,000003,2025,1,exact
3,000005,2023,2,conflict
4,000006,2024,1,exact
5,000008,2022,2,conflict
6,000009,2023,1,exact
7,000011,2021,2,conflict
8,000012,2022,1,exact
9,000014,2020,2,conflict


In [29]:
duplicate_group_classification[
    "duplicate_type"
].value_counts()

duplicate_type
exact       10
conflict    10
Name: count, dtype: int64

In [30]:
conflict_group_mask = (
    duplicate_group_classification["duplicate_type"]
    == "conflict"
)

conflict_variation = group_variation.loc[
    conflict_group_mask
]

conflict_summary = (
    conflict_variation
    .gt(1)
    .apply(
        lambda row: ", ".join(
            row.index[row].tolist()
        ),
        axis=1,
    )
    .rename("different_columns")
    .reset_index()
)

conflict_summary

,stock_code,year,different_columns
0,000002,2024,net_profit
1,000005,2023,net_profit
2,000008,2022,net_profit
3,000011,2021,net_profit
4,000014,2020,net_profit
5,000016,2025,net_profit
6,000019,2024,net_profit
7,000022,2023,net_profit
8,000025,2022,net_profit
9,000028,2021,net_profit


In [31]:
duplicate_rows_classified = (
    duplicate_firm_year_rows
    .merge(
        duplicate_group_classification[
            ["duplicate_type"]
        ].reset_index(),
        on=key_cols,
        how="left",
        validate="many_to_one",
    )
)

duplicate_rows_classified[
    "duplicate_type"
].value_counts()

duplicate_type
exact       20
conflict    20
Name: count, dtype: int64

## 第四阶段：安全去除完全重复记录

在 20 个重复 firm-year 组中：

- 10 个属于完全重复；
- 10 个属于冲突重复；
- 冲突重复目前全部只在 `net_profit` 上存在不同取值。

本阶段只删除完全重复记录中的多余副本。

冲突重复记录全部保留，后续单独制定处理规则。

In [32]:
exact_duplicate_later_mask = financials.duplicated(
    keep="first"
)

rows_to_remove = financials.loc[
    exact_duplicate_later_mask
].copy()

print("准备删除的完全重复记录数:", len(rows_to_remove))

准备删除的完全重复记录数: 10


In [33]:
rows_to_remove_check = rows_to_remove.merge(
    duplicate_group_classification[
        ["duplicate_type"]
    ].reset_index(),
    on=key_cols,
    how="left",
    validate="many_to_one",
)

print(
    rows_to_remove_check[
        "duplicate_type"
    ].value_counts(dropna=False)
)

rows_to_remove_check[
    [
        "stock_code",
        "year",
        "duplicate_type",
    ]
]

duplicate_type
exact    10
Name: count, dtype: int64


,stock_code,year,duplicate_type
0,000001,2020,exact
1,000003,2025,exact
2,000006,2024,exact
3,000009,2023,exact
4,000012,2022,exact
5,000015,2021,exact
6,000018,2020,exact
7,000020,2025,exact
8,000023,2024,exact
9,000026,2023,exact


In [34]:
financials_after_exact_dedup = financials.loc[
    ~exact_duplicate_later_mask
].copy()

print("删除前记录数:", len(financials))
print("删除后记录数:", len(financials_after_exact_dedup))
print(
    "实际减少记录数:",
    len(financials) - len(financials_after_exact_dedup),
)

删除前记录数: 260
删除后记录数: 250
实际减少记录数: 10


In [35]:
remaining_firm_year_duplicate_mask = (
    financials_after_exact_dedup.duplicated(
        subset=key_cols,
        keep=False,
    )
)

remaining_duplicate_rows = (
    financials_after_exact_dedup.loc[
        remaining_firm_year_duplicate_mask
    ].copy()
)

remaining_duplicate_group_count = (
    remaining_duplicate_rows
    .groupby(key_cols, dropna=False)
    .ngroups
)

remaining_exact_duplicate_count = (
    financials_after_exact_dedup
    .duplicated(keep="first")
    .sum()
)

print(
    "剩余重复 firm-year 记录数:",
    remaining_firm_year_duplicate_mask.sum(),
)

print(
    "剩余重复 firm-year 组数:",
    remaining_duplicate_group_count,
)

print(
    "剩余完全重复后续记录数:",
    remaining_exact_duplicate_count,
)

剩余重复 firm-year 记录数: 20
剩余重复 firm-year 组数: 10
剩余完全重复后续记录数: 0


In [36]:
conflict_keys_before = set(
    map(
        tuple,
        duplicate_group_classification
        .query("duplicate_type == 'conflict'")
        .reset_index()[key_cols]
        .to_numpy(),
    )
)

remaining_duplicate_keys = set(
    map(
        tuple,
        remaining_duplicate_rows[
            key_cols
        ]
        .drop_duplicates()
        .to_numpy(),
    )
)

print("原 conflict 组数:", len(conflict_keys_before))
print("删除后剩余重复组数:", len(remaining_duplicate_keys))

print(
    "剩余重复组是否恰好等于原 conflict 组:",
    conflict_keys_before == remaining_duplicate_keys,
)

原 conflict 组数: 10
删除后剩余重复组数: 10
剩余重复组是否恰好等于原 conflict 组: True


In [37]:
assert len(rows_to_remove) == 10

assert (
    rows_to_remove_check["duplicate_type"]
    .eq("exact")
    .all()
)

assert len(financials_after_exact_dedup) == 250

assert remaining_exact_duplicate_count == 0

assert (
    remaining_firm_year_duplicate_mask.sum()
    == 20
)

assert remaining_duplicate_group_count == 10

assert (
    conflict_keys_before
    == remaining_duplicate_keys
)

financials = financials_after_exact_dedup

print("完全重复记录清理完成")
print("当前记录数:", len(financials))

完全重复记录清理完成
当前记录数: 250


## 第五阶段：财务数值字段审计

完成主键标准化和完全重复去重后，开始检查财务数值字段。

本阶段先识别：
- 哪些列已经是数值类型；
- 哪些列因为特殊字符串而变成 object；
- 是否存在逗号、百分号、伪缺失值等格式问题。

暂不进行实际转换。

In [38]:
numeric_candidate_cols = [
    "total_assets",
    "total_liabilities",
    "revenue",
    "net_profit",
    "cash",
    "rd_expense",
    "roe",
    "employees",
]

financials[numeric_candidate_cols].dtypes

total_assets          object
total_liabilities     object
revenue               object
net_profit           float64
cash                 float64
rd_expense            object
roe                   object
employees            float64
dtype: object

In [39]:
for col in numeric_candidate_cols:
    string_mask = financials[col].map(
        lambda x: isinstance(x, str)
    )

    string_values = financials.loc[
        string_mask,
        col,
    ]

    print("=" * 60)
    print("变量:", col)
    print("字符串记录数:", len(string_values))

    if len(string_values) > 0:
        print(
            string_values
            .value_counts(dropna=False)
            .head(20)
        )

变量: total_assets
字符串记录数: 1
total_assets
31,241,347,993    1
Name: count, dtype: int64
变量: total_liabilities
字符串记录数: 1
total_liabilities
-    1
Name: count, dtype: int64
变量: revenue
字符串记录数: 1
revenue
15,335,659,776    1
Name: count, dtype: int64
变量: net_profit
字符串记录数: 0
变量: cash
字符串记录数: 0
变量: rd_expense
字符串记录数: 1
rd_expense
-    1
Name: count, dtype: int64
变量: roe
字符串记录数: 2
roe
13.5%    1
7.25%    1
Name: count, dtype: int64
变量: employees
字符串记录数: 0


In [40]:
numeric_missing_summary = pd.DataFrame(
    {
        "dtype": financials[
            numeric_candidate_cols
        ].dtypes.astype(str),

        "real_missing": financials[
            numeric_candidate_cols
        ].isna().sum(),
    }
)

numeric_missing_summary

,dtype,real_missing
total_assets,object,0
total_liabilities,object,0
revenue,object,0
net_profit,float64,1
cash,float64,1
rd_expense,object,0
roe,object,0
employees,float64,1


In [41]:
for col in ["rd_expense", "roe"]:
    string_mask = financials[col].map(
        lambda x: isinstance(x, str)
    )

    string_values = financials.loc[
        string_mask,
        col,
    ]

    print("=" * 50)
    print("变量:", col)
    print("字符串记录数:", len(string_values))
    print("字符串取值及数量:")
    print(string_values.value_counts(dropna=False))

变量: rd_expense
字符串记录数: 1
字符串取值及数量:
rd_expense
-    1
Name: count, dtype: int64
变量: roe
字符串记录数: 2
字符串取值及数量:
roe
13.5%    1
7.25%    1
Name: count, dtype: int64


In [42]:
roe_numeric_mask = financials["roe"].map(
    lambda x: isinstance(x, (int, float))
)

roe_numeric = financials.loc[
    roe_numeric_mask,
    "roe",
]

print("数值型 ROE 记录数:", len(roe_numeric))
print("最小值:", roe_numeric.min())
print("最大值:", roe_numeric.max())

print("\n绝对值最大的 15 条数值型 ROE:")
print(
    roe_numeric
    .abs()
    .sort_values(ascending=False)
    .head(15)
)

数值型 ROE 记录数: 248
最小值: -1.9169
最大值: 38.593

绝对值最大的 15 条数值型 ROE:
91     38.593
58       13.5
211    5.8785
139    4.9073
40     3.3773
35     2.1835
10     2.0943
250    2.0943
42     1.9169
130    1.7754
236    1.6603
252    1.5301
44     1.5301
205    1.3511
111    1.3407
Name: roe, dtype: object


## 阶段性结论：2026-09-17

本阶段已完成财务面板训练数据的第一轮正式清洗与数值字段审计。

已完成：

- `stock_code` 标准化为 6 位字符串，并保留原有缺失；
- `year` 标准化为 pandas `Int64` 年份；
- 识别 20 个重复 firm-year 组；
- 将重复组区分为 10 个完全重复组和 10 个冲突重复组；
- 安全删除 10 条完全重复的多余副本，记录数由 260 条降至 250 条；
- 保留全部冲突 firm-year，不进行无依据删除；
- 已确认剩余 10 个冲突组均只在 `net_profit` 上存在不同取值；
- 完成主要财务数值字段的格式审计。

当前已识别的数值字段问题包括：

- `total_assets`：存在千位逗号格式；
- `revenue`：存在千位逗号格式；
- `total_liabilities`：存在 `"-"` 伪缺失值；
- `rd_expense`：存在 `"-"` 伪缺失值；
- `net_profit`、`cash`、`employees`：存在真实缺失值；
- `roe`：同时存在数值形式与百分号字符串形式，并存在需要进一步判断的异常或尺度问题。

本阶段暂不强行处理冲突 firm-year 和 ROE，后续将在明确规则后继续清洗。

原始数据文件始终保持不修改。